# Initial Proof of Concept 

## Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from scipy import stats
import ipywidgets as widgets
from IPython.display import display, clear_output

# styling
sns.set_theme(style="white")
plt.rcParams["figure.figsize"] = (12, 5)

## Core Functions

In [2]:
def fetch_and_process_data(ticker, period_code):
    """
    fetches OHLCV data from yfinance and computes daily returns.
    """
    periods = {"1": "1mo", "2": "3mo", "3": "6mo", "4": "1y", "5": "5y", "6": "max"}
    chosen_period = periods.get(period_code, "1y")

    data = yf.download(ticker, period=chosen_period, auto_adjust=True, progress=False)
    if data.empty:
        raise ValueError(f"No data found for ticker '{ticker}'. Check the symbol and try again.")

    data['Return'] = data['Close'].pct_change()
    data['Log_Return'] = np.log(data['Close'] / data['Close'].shift(1))
    data.dropna(inplace=True)
    return data, chosen_period

In [3]:
def run_risk_analysis(data, num_simulations=10000, forecast_days=21, seed=None):
    """
    runs historical VaR, CVaR, and a Monte Carlo simulation using bootstrap
    resampling (not GBM). returns a metrics dict.

    used bootstrap resampling to preserving fat tails and skew without any distributional assumption.
    """
    if seed is not None:
        np.random.seed(seed)

    returns = data['Return'].squeeze().values
    last_price = float(data['Close'].squeeze().iloc[-1])

    # VaR and CVaR
    hist_var_95 = np.percentile(returns, 5)
    hist_var_99 = np.percentile(returns, 1)
    hist_cvar_95 = float(returns[returns <= hist_var_95].mean())
    hist_cvar_99 = float(returns[returns <= hist_var_99].mean())

    # bootstrap monte carlo ---
    # avoids the normal distribution assumption
    simulated_daily_returns = np.random.choice(
        returns, size=(num_simulations, forecast_days), replace=True
    )

    # price paths: P_t = P_0 * cumprod(1 + r_i)
    simulated_price_paths = last_price * (1 + simulated_daily_returns).cumprod(axis=1)

    # horizon returns at end of forecast window
    horizon_returns = (simulated_price_paths[:, -1] - last_price) / last_price
    mc_horizon_var_95 = np.percentile(horizon_returns, 5)
    mc_horizon_var_99 = np.percentile(horizon_returns, 1)
    mc_horizon_cvar_95 = float(horizon_returns[horizon_returns <= mc_horizon_var_95].mean())
    mc_horizon_cvar_99 = float(horizon_returns[horizon_returns <= mc_horizon_var_99].mean())

    # daily MC metrics
    all_sim_daily = simulated_daily_returns.flatten()
    mc_daily_var_95 = np.percentile(all_sim_daily, 5)
    mc_daily_var_99 = np.percentile(all_sim_daily, 1)
    mc_daily_cvar_95 = float(all_sim_daily[all_sim_daily <= mc_daily_var_95].mean())  # add
    mc_daily_cvar_99 = float(all_sim_daily[all_sim_daily <= mc_daily_var_99].mean())  # add

    return {
        # historical
        "hist_var_95": hist_var_95, "hist_var_99": hist_var_99,
        "hist_cvar_95": hist_cvar_95, "hist_cvar_99": hist_cvar_99,
        # MC daily
        "mc_daily_var_95": mc_daily_var_95, "mc_daily_var_99": mc_daily_var_99,
        "mc_daily_cvar_95": mc_daily_cvar_95, "mc_daily_cvar_99": mc_daily_cvar_99,
        # MC horizon
        "mc_horizon_var_95": mc_horizon_var_95, "mc_horizon_var_99": mc_horizon_var_99,
        "mc_horizon_cvar_95": mc_horizon_cvar_95, "mc_horizon_cvar_99": mc_horizon_cvar_99,
        # paths
        "sim_paths": simulated_price_paths,
        "horizon_returns": horizon_returns,
        "last_price": last_price,
        "forecast_days": forecast_days,
    }

In [4]:
def backtest_var(returns, hist_var_95, hist_var_99):
    """
    checks how often historical returns actually breached the VaR thresholds.
    significant deviation suggests model mis-calibration.
    """
    n = len(returns)
    breaches_95 = int((returns < hist_var_95).sum())
    breaches_99 = int((returns < hist_var_99).sum())
    return {
        "n": n,
        "breaches_95": breaches_95, "rate_95": breaches_95 / n,
        "breaches_99": breaches_99, "rate_99": breaches_99 / n,
        "expected_95": 0.05, "expected_99": 0.01,
    }

## Visualization

In [5]:
def plot_dashboard(data, metrics, bt, ticker):
    """
    3-panel dashboard:
      left return distribution with historical VaR/CVaR lines
      middle monte carlo price trajectories
      right backtest breach rate bar chart
    """
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 6))

    # return distribution
    returns = data['Return'].squeeze()
    sns.histplot(returns, kde=True, color="steelblue", stat="density", ax=ax1)
    ax1.axvline(metrics["hist_var_95"], color="orange", linestyle="--", linewidth=2,
                label=f"Hist 95% VaR ({metrics['hist_var_95']:.2%})")
    ax1.axvline(metrics["hist_var_99"], color="red", linestyle="--", linewidth=2,
                label=f"Hist 99% VaR ({metrics['hist_var_99']:.2%})")
    ax1.axvline(metrics["hist_cvar_95"], color="orange", linestyle=":", linewidth=1.5,
                label=f"Hist 95% CVaR ({metrics['hist_cvar_95']:.2%})")
    ax1.axvline(metrics["hist_cvar_99"], color="darkred", linestyle=":", linewidth=1.5,
                label=f"Hist 99% CVaR ({metrics['hist_cvar_99']:.2%})")
    ax1.set_title(f"{ticker} — Daily Return Distribution")
    ax1.set_xlabel("Daily Return")
    ax1.legend(fontsize=8)

    # monte carlo price paths 
    paths_to_plot = metrics["sim_paths"][:200]
    time_horizon = np.arange(1, metrics["forecast_days"] + 1)
    for path in paths_to_plot:
        ax2.plot(time_horizon, path, color="grey", alpha=0.05)
    mean_path = np.mean(metrics["sim_paths"], axis=0)
    p5_path   = np.percentile(metrics["sim_paths"], 5, axis=0)
    p1_path   = np.percentile(metrics["sim_paths"], 1, axis=0)
    ax2.plot(time_horizon, mean_path, color="royalblue", linewidth=2, label="Mean Path")
    ax2.plot(time_horizon, p5_path,   color="orange",   linewidth=2, linestyle="-.", label="5th Pctile")
    ax2.plot(time_horizon, p1_path,   color="red",      linewidth=2, linestyle=":",  label="1st Pctile")
    ax2.axhline(metrics["last_price"], color="black", linewidth=1, linestyle="--", alpha=0.4, label="Current Price")
    ax2.set_title(f"Bootstrap MC: {len(paths_to_plot)} Sample Paths ({metrics['forecast_days']} Days)")
    ax2.set_xlabel("Trading Days Forward")
    ax2.set_ylabel("Projected Price ($)")
    ax2.legend(fontsize=8)

    # backtest breach rates 
    labels = ["95% VaR", "99% VaR"]
    actual   = [bt["rate_95"] * 100, bt["rate_99"] * 100]
    expected = [bt["expected_95"] * 100, bt["expected_99"] * 100]
    x = np.arange(len(labels))
    width = 0.35
    bars_actual   = ax3.bar(x - width/2, actual,   width, label="Actual Breach %",   color="steelblue")
    bars_expected = ax3.bar(x + width/2, expected, width, label="Expected Breach %", color="lightcoral")
    ax3.set_title("VaR Backtest — Breach Rates")
    ax3.set_xticks(x)
    ax3.set_xticklabels(labels)
    ax3.set_ylabel("Breach Rate (%)")
    ax3.legend(fontsize=8)
    for bar in bars_actual:
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f"{bar.get_height():.1f}%", ha='center', va='bottom', fontsize=9)
    for bar in bars_expected:
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f"{bar.get_height():.1f}%", ha='center', va='bottom', fontsize=9)

    plt.suptitle(f"{ticker} — Market Risk Dashboard", fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

## Dashboard UI

In [6]:
ticker_input = widgets.Text(
    value='SPY', description='Ticker:', style={'description_width': 'initial'}
)
period_dropdown = widgets.Dropdown(
    options=[('1 Month', '1'), ('3 Months', '2'), ('6 Months', '3'),
             ('1 Year', '4'), ('5 Years', '5'), ('Max', '6')],
    value='4', description='History:',
    style={'description_width': 'initial'}
)
horizon_slider = widgets.IntSlider(
    value=21, min=5, max=252, step=1,
    description='Forecast Days:',
    style={'description_width': 'initial'}
)
sims_dropdown = widgets.Dropdown(
    options=[('5,000 (fast)', 5000), ('10,000 (balanced)', 10000),
             ('50,000 (precise)', 50000), ('100,000 (slow)', 100000)],
    value=10000, description='Simulations:',
    style={'description_width': 'initial'}
)
run_button  = widgets.Button(description="Generate Dashboard", button_style='primary', icon='check')
output_area = widgets.Output()


def on_button_clicked(b):
    with output_area:
        clear_output(wait=True)
        tk = ticker_input.value.upper().strip()
        if not tk:
            print("Please enter a ticker symbol.")
            return
        try:
            # data
            df, period_str = fetch_and_process_data(tk, period_dropdown.value)
            n_rows = len(df)
            print(f"Data loaded: {tk} ({period_str}) — {n_rows} trading days")
            if n_rows < 60:
                print(f"⚠  Warning: only {n_rows} observations. VaR estimates are unreliable "
                      "with fewer than ~60 data points. Consider a longer period.")

            # risk model
            results = run_risk_analysis(
                df,
                num_simulations=sims_dropdown.value,
                forecast_days=horizon_slider.value
            )
            returns = df['Return'].squeeze()
            bt = backtest_var(returns.values, results['hist_var_95'], results['hist_var_99'])

            # summary stats
            ann_ret = returns.mean() * 252
            ann_vol = returns.std() * np.sqrt(252)
            sharpe  = ann_ret / ann_vol if ann_vol != 0 else float('nan')
            skew    = float(returns.skew())
            kurt    = float(returns.kurtosis())

            print(f"\n{'─'*55}")
            print(f"  Summary Statistics")
            print(f"{'─'*55}")
            print(f"  Current Price        : ${results['last_price']:>10.2f}")
            print(f"  Annualised Return    : {ann_ret:>10.2%}")
            print(f"  Annualised Volatility: {ann_vol:>10.2%}")
            print(f"  Sharpe Ratio         : {sharpe:>10.2f}  (assumes 0% risk-free rate)")
            print(f"  Skewness             : {skew:>10.4f}")
            print(f"  Excess Kurtosis      : {kurt:>10.4f}")

            # daily VaR / CVaR table 
            print(f"\n{'─'*65}")
            print(f"  Daily Risk Metrics (Historical vs Bootstrap MC)")
            print(f"{'─'*65}")
            print(f"  {'Metric':<22} {'Historical':>12} {'MC Daily':>12} {'Diff':>10}")
            print(f"  {'─'*60}")
            for label, h, mc in [
                ("95% VaR",  results['hist_var_95'],  results['mc_daily_var_95']),
                ("99% VaR",  results['hist_var_99'],  results['mc_daily_var_99']),
                ("95% CVaR", results['hist_cvar_95'], results['mc_daily_cvar_95']),
                ("99% CVaR", results['hist_cvar_99'], results['mc_daily_cvar_99']),
            ]:
                mc_str   = f"{mc:.4%}" if mc is not None else "       —"
                diff_str = f"{mc - h:.4%}" if mc is not None else "       —"
                print(f"  {label:<22} {h:>12.4%} {mc_str:>12} {diff_str:>10}")

            # horizon VaR / CVaR 
            fd = results['forecast_days']
            print(f"\n{'─'*55}")
            print(f"  {fd}-Day Horizon Risk (Bootstrap MC)")
            print(f"{'─'*55}")
            print(f"  95% Horizon VaR  : {results['mc_horizon_var_95']:>10.2%}")
            print(f"  99% Horizon VaR  : {results['mc_horizon_var_99']:>10.2%}")
            print(f"  95% Horizon CVaR : {results['mc_horizon_cvar_95']:>10.2%}")
            print(f"  99% Horizon CVaR : {results['mc_horizon_cvar_99']:>10.2%}")

            # backtest 
            print(f"\n{'─'*55}")
            print(f"  VaR Backtest (n = {bt['n']} observations)")
            print(f"{'─'*55}")
            for label, actual, expected, count in [
                ("95% VaR", bt['rate_95'], bt['expected_95'], bt['breaches_95']),
                ("99% VaR", bt['rate_99'], bt['expected_99'], bt['breaches_99']),
            ]:
                flag = "X" if actual > expected * 2 or actual < expected * 0.5 else "✓"
                print(f"  {label}: {actual:.2%} actual vs {expected:.0%} expected "
                      f"({count} breaches){flag}")

            # charts 
            plot_dashboard(df, results, bt, tk)

        except Exception as e:
            print(f"Error: {e}")


run_button.on_click(on_button_clicked)

In [7]:
ui_box = widgets.VBox([
    widgets.HTML(value="<h2 style='margin-bottom:8px'>Placeholder (send name ideas)</h2>"
                       "<p style='color:#555;font-size:13px'>"
                       "Bootstrap Monte Carlo · Historical VaR/CVaR · Backtest</p>"),
    widgets.HBox([ticker_input, period_dropdown]),
    widgets.HBox([horizon_slider, sims_dropdown]),
    widgets.Box([run_button], layout=widgets.Layout(margin='10px 0px')),
    output_area
])
display(ui_box)